In [ ]:
import numpy
import pandas as pd
import numpy as np
import tensorflow as tf
from keras import layers, Input, models
from keras.models import Model
from keras.layers import Reshape, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from keras.utils import plot_model

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

seed_value = 42
tf.random.set_seed(seed_value)
np.random.seed(seed_value)


# Function to load input (descriptor vectors) from CSV
def load_input_data(csv_file,rand):
    df = pd.read_csv(csv_file, header=0)  # Assuming no header in CSV
    if rand > 0 :
        df=df.sample(n=rand)
    return np.array(df.iloc[:, 1:-6].astype('float32')), np.array(df.iloc[:, -6:].astype('int'))

def load_input_data_smi(csv_file,rand):
    df = pd.read_csv(csv_file, header=0)  # Assuming no header in CSV
    if rand > 0 :
        df=df.sample(n=rand)
    return np.array(df.iloc[:, 1:-7].astype('float32')), \
        np.array(df.iloc[:, -7:-1].astype('int')),\
        np.array(df.iloc[:, -1].astype('string'))

# Load input and output data from CSV files
input_csv_file = 'C:/Users/mohammed/Documents/NNV/research/opioid_NN/suma/codes/oprpred/data/Data/union/Training&validation_data_pentacle fingerprints_smiles_csv.csv'

input_csv_file_pubchem = 'C:/Users/mohammed/Documents/NNV/research/opioid_NN/suma/codes/oprpred/data/Data/union/Training&validation_data_pubchem_penta_filtered fingerprints_csv.csv'

#input_csv_file_pubchem = 'C:/Users/mohammed/Documents/NNV/research/opioid_NN/suma/codes/oprpred/data/Data/union/Training&validation_data_ECFP_penta_filtered fingerprints_csv.csv'
#input_csv_file = 'C:/Users/mohammed/Documents/NNV/research/opioid_NN/suma/codes/oprpred/data/Data/union/Training data_pubchem fingerprints_csv.csv'


#X,y = load_input_data(input_csv_file,0)  # Load input (descriptor vectors)

X,y,z= load_input_data_smi(input_csv_file,0)  # Load input (descriptor vectors)
## convert X into X^2
#X=np.square(X)
X2,y2,z2= load_input_data_smi(input_csv_file_pubchem,0)  # Load input (descriptor vectors)
# mormalized matrix
X = (X - X.min()) / (X.max() - X.min())

# Alternative Z-score normalization(standardization) Standardize data to mean 0 and standard deviation 1
# Compute mean and std along columns (features)
# mean = np.mean(X, axis=0)
# std = np.std(X, axis=0)
# std[std == 0] = 1e-8
# X = (X - mean) / std
X

# Handle zero std: replace 0 with a small value (e.g., 1e-8)


# Standardize the data
X
y_all=y[(y[:,0]!=1) & (y[:,1]!=1)& (y[:,2]!=1)& (y[:,3]!=1)& (y[:,4]!=1)& (y[:,5]!=1)]
y_all
#         0                      1                        2
#OPRD.agonist.outcome(0.7)	OPRK.agonist.outcome(0.76)	OPRM.agonist.outcome (0.80)

#          3                           4                              5
#OPRD.antagonist.outcome(0.72)	OPRK.antagonist.outcome(0.77)	OPRM.antagonist.outcome(0.66)


selected_activity=0
vector_size=X.shape[1]

y_activity=y[:,selected_activity]
y_activity
y_all_filtered=y[(y[:,0]!=1) & (y[:,1]!=1)& (y[:,2]!=1)& (y[:,3]!=1)& (y[:,4]!=1)& (y[:,5]!=1)]
y_all_filtered[y_all_filtered==2]=1
X_all_filtered=X[(y[:,0]!=1) & (y[:,1]!=1)& (y[:,2]!=1)& (y[:,3]!=1)& (y[:,4]!=1)& (y[:,5]!=1)]



###filter Y and X
### CHANGE 2 TO 1 IN ACTIVITY
y_activtiy_filtered=y_activity[y_activity[:] != 1]

#convert 2 to 1 for actives, while keep inactive as 0
y_activtiy_filtered[y_activtiy_filtered== 2]=1

X_filtered=X[y_activity[:] != 1]
X2_filtered=X2[y_activity[:] != 1]
z_filtered=z[y_activity[:] != 1]
y_activtiy_filtered

X=X_filtered
X2=X2_filtered
y=y_activtiy_filtered


X_all_filtered
X2_filtered
y_activtiy_filtered

#####Y randomized ######
#######################
#np.random.shuffle(y)


# moh
# used to cut part of pentacle vector either from short bins side (left restriction) or from larger bin side (right restriction)
right_restriction=130
left_restriction=0
cutted_X=[]
for row in X_filtered:
    # Step 2: Reshape the row into a 10x50 array
    reshaped_row = row.reshape(10, 187)

    # Step 3: Remove the first 3 columns
    reshaped_row = reshaped_row[:, 0+left_restriction:-1-right_restriction]

    # Step 4: Reshape back to a 470-element row
    modified_row = reshaped_row.flatten()
    cutted_X.append(modified_row)
cutted_X=np.array(cutted_X)
vector_size=cutted_X.shape[1]
fingerprint_size=X2.shape[1]

X=cutted_X
#y=y_activtiy_filtered
print(vector_size)
print(fingerprint_size)
###### only used in case of no 5-fold cross validation)
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

# Define the Y-axis labels
y_labels = ["Dry-Dry", "O-O", "N1-N1", "TIP-TIP", "Dry-O",
            "Dry-N1", "Dry-TIP", "O-N1", "O-TIP", "N1-TIP"]


def plot_input_matrix (input_vector):
    # Reshape to 10 rows
        reshaped_data = input_vector.reshape(10, int(vector_size / 10))

        # Create the plot with the same size and styling as the saliency map
        fig, ax = plt.subplots(figsize=(10, 5))  # Match the figure size
        #im = ax.imshow(reshaped_data, cmap='gray', aspect='auto', interpolation='bilinear', vmin=np.min(X[i]), vmax=np.max(X[i]))  # Use 'jet' colormap and bilinear interpolation
        im = plt.imshow(reshaped_data, cmap='gray', aspect='auto',vmin=np.min(X[i]), vmax=np.max(X[i]))  # Use 'jet' for a heatmap look

        # Add a shorter color bar
        cbar = plt.colorbar(im, fraction=0.046, pad=0.04)  # Match the color bar size
        cbar.set_label("Importance")  # Add label to color bar

        # Set y-axis labels
        ax.set_yticks(np.arange(10))  # 10 rows
        ax.set_yticklabels(y_labels)

        # Label the axes
        ax.set_xlabel("Feature Columns")
        ax.set_ylabel("Feature Groups")

        # Set the title
        plt.title(f"Feature Finger Print Map for Sample {i+1}")

        plt.show()

for i in range(4):  # Loop through 4 samples
    if i < 400:
        plot_input_matrix(X[i])

from keras.layers import Masking
from tensorflow.keras.layers import Layer
from tensorflow.keras.initializers import Initializer
from tensorflow.keras.backend import expand_dims



def create_2D_cnn_regression2(input_shape=(vector_size,), output_shape=1, num_classes=1):
    model = models.Sequential()
    # mohammedNoor2
    # Reshape the input to a 2D image-like structure (height=1, width=600, channels=1)
    model.add(layers.Reshape((10, int(vector_size/10), 1), input_shape=input_shape))

    #model.add(WeightedAveragePooling(pool_size=2, strides=2))
   # model.add(ProgressiveColumnWeighting())
   ## model.add(ProgressiveColumnWeightingExp(growth_rate=1.05))
     # Apply progressively weighted pooling
    #model.add(ProgressiveWeightedPooling(pool_size=2, strides=2))

    # Apply Progressive Column Attention
    #model.add(ProgressiveColumnAttention())

    model.add( Masking(mask_value=0.0, input_shape=(10, int(vector_size/10))))
    # Convolutional layers
    # Apply max pooling across every two columns


    # Apply mean pooling across every two columns
    # model.add(layers.AveragePooling2D(pool_size=(1, 5), strides=(1, 5), padding='valid'))  # Pool only across the column axis
   # model.add(layers.MaxPooling2D(pool_size=(1, 5), strides=(1, 5), padding='valid'))  # Pool only across the column axis




    model.add(layers.Conv2D(3, (10,10), activation=None,dilation_rate=(1,1), padding='same', strides=(1, 1),kernel_regularizer=tf.keras.regularizers.l2(0.1),kernel_initializer='he_normal'))

    model.add(layers.BatchNormalization())
    model.add(layers.Activation('leaky_relu'))  # Apply activation after Batch Norm
    model.add(layers.Dropout(0.15))
    model.add(layers.MaxPooling2D((1, 4)))
   # Apply attention after the first block
   # model.add(SpatialAttention())

    model.add(layers.Conv2D(3, (10,10), activation=None,dilation_rate=(1,1), padding='same', strides=(1, 1),kernel_regularizer=tf.keras.regularizers.l2(0.1),kernel_initializer='he_normal'))

    model.add(layers.BatchNormalization())
    model.add(layers.Activation('leaky_relu'))  # Apply activation after Batch Norm
    model.add(layers.Dropout(0.15))
    model.add(layers.MaxPooling2D((1, 4)))
    #Apply attention after the first block
    #model.add(SpatialAttention())



   # model.add(layers.Conv2D(2, (10,10), activation=None,dilation_rate=(1,1), padding='same', strides=(1, 1),kernel_regularizer=tf.keras.regularizers.l2(0.01),kernel_initializer='he_normal'))

   # model.add(layers.BatchNormalization())
   # model.add(layers.Activation('leaky_relu'))  # Apply activation after Batch Norm
   # model.add(layers.Dropout(0.35))
   # model.add(layers.MaxPooling2D((1, 2)))
   # Apply attention after the first block
   # model.add(SpatialAttention())

    #model.add(layers.Conv2D(5, (5,5), activation=None,dilation_rate=(1,1), padding='same', strides=(1, 1),kernel_regularizer=tf.keras.regularizers.l2(0.01),kernel_initializer='he_normal'))

    #model.add(layers.BatchNormalization())
    #model.add(layers.Activation('leaky_relu'))  # Apply activation after Batch Norm
    #model.add(layers.Dropout(0.35))
    #model.add(layers.MaxPooling2D((1, 2)))
   # Apply attention after the first block
   # model.add(SpatialAttention())


    # Flatten the output to feed into a fully connected layer
    model.add(layers.Flatten())

    #Dense (fully connected) layers for regression
    model.add(layers.Dense(10, activation=None,kernel_regularizer=tf.keras.regularizers.l2(0.1), kernel_initializer='he_normal'))   # , kernel_initializer='glorot_uniform'
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('leaky_relu'))  # Apply activation after Batch Norm
    model.add(layers.Dropout(0.15))




    model.add(layers.Dense(output_shape * num_classes, activation='sigmoid'))  # No activation function for regression
    #model.add(layers.Reshape((output_shape, num_classes)))
    return model

import tensorflow as tf
from tensorflow.keras import layers, models

def create_2D_cnn_regression2_(input_shape=(1870,), output_shape=1, num_classes=1):
    inputs = layers.Input(shape=input_shape)


    # Reshape input into 2D matrix: (10 rows, 187 columns, 1 channel)
    reshaped = layers.Reshape((10, int(input_shape[0] / 10), 1))(inputs)

    # Trainable Row Importance Mechanism (10 Neurons for 10 Rows)
    row_features = tf.reduce_mean(reshaped, axis=-1)  # Reduce across channels to get (batch_size, 10, columns)
    row_features = tf.reduce_mean(row_features, axis=-1)  # Further reduce across columns to get (batch_size, 10)

    row_scores = layers.Dense(10, activation='softmax', name="row_importance")(row_features)  # 10 neurons (one per row)
    row_scores = layers.Reshape((10, 1, 1))(row_scores)  # Correctly reshape for broadcasting
    weighted_inputs = layers.Multiply()([reshaped, row_scores])  # Apply learned row scores



    # First Convolutional Block
    x = layers.Conv2D(3, (10,10), activation=None, padding='same', strides=(1, 1),
                      kernel_regularizer=tf.keras.regularizers.l2(0.035),kernel_initializer='he_normal')(weighted_inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('leaky_relu')(x)
    x = layers.Dropout(0.15)(x)
    x = layers.MaxPooling2D((1, 4))(x)

    # Second Convolutional Block
    x = layers.Conv2D(3, (10,10), activation=None, padding='same', strides=(1, 1),
                      kernel_regularizer=tf.keras.regularizers.l2(0.035), kernel_initializer='he_normal')(weighted_inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('leaky_relu')(x)
    x = layers.Dropout(0.15)(x)
    x = layers.MaxPooling2D((1, 4))(x)



    # Fully Connected Layer
    x = layers.Flatten()(x)
    x = layers.Dense(10, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.035), kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('leaky_relu')(x)
    x = layers.Dropout(0.15)(x)

    # Output Layer
    outputs = layers.Dense(output_shape * num_classes, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)
    return model



def create_2D_cnn_regression3(input_shape=(vector_size,), output_shape=1, num_classes=1):
    model = models.Sequential()

    # Reshape the input to a 2D image-like structure (height=1, width=600, channels=1)
    model.add(layers.Reshape((10, int(vector_size/10), 1), input_shape=input_shape))
    #model.add(Masking(mask_value=0.0, input_shape=(10, int(vector_size/10))))
    # Convolutional layers
    model.add(layers.Conv2D(3, (10, 1), activation=None, padding='same', strides=(1, 1),kernel_regularizer=tf.keras.regularizers.l2(0.03)))

    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))  # Apply activation after Batch Norm
    model.add(layers.Dropout(0.25))
    model.add(layers.MaxPooling2D((1, 4)))

    model.add(layers.Conv2D(5, (1,int(vector_size/10)), activation=None, padding='same', strides=(1, 1),kernel_regularizer=tf.keras.regularizers.l2(0.03)))
    model.add(layers.BatchNormalization())

    model.add(layers.Activation('relu'))  # Apply activation after Batch Norm
    model.add(layers.Dropout(0.25))
    model.add(layers.MaxPooling2D((1, 5)))


    # Flatten the output to feed into a fully connected layer
    model.add(layers.Flatten())

    # Use DepthwiseConv2D as a fully connected layer to mix features across channels
    #model.add(layers.DepthwiseConv2D((1, 1), activation='relu'))


    #model.add(layers.GlobalAveragePooling2D())
    # Dense (fully connected) layers for regression
    model.add(layers.Dense(10, activation=None,kernel_regularizer=tf.keras.regularizers.l2(0.03)))   # , kernel_initializer='glorot_uniform'
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))  # Apply activation after Batch Norm
    model.add(layers.Dropout(0.25))




    model.add(layers.Dense(output_shape * num_classes, activation='sigmoid'))  # No activation function for regression
    #model.add(layers.Reshape((output_shape, num_classes)))
    return model

#mohammednoor
def create_2D_cnn_regression_parallel(vector_size=vector_size, fingerprint_size=fingerprint_size, output_shape=1, num_classes=1):
    total_size = vector_size + fingerprint_size
    full_input = Input(shape=(total_size,), name="full_input")

    # Proper symbolic slicing using Lambda
    input1 = layers.Lambda(lambda x: x[:, :vector_size], name="grind_slice")(full_input)
    input2 = layers.Lambda(lambda x: x[:, vector_size:], name="fp_slice")(full_input)
    #inputs = full_input[:, :560]
    #input2 = full_input[:, 560:]

    # Reshape the input to a 2D image-like structure (height=1, width=600, channels=1)
    reshaped = layers.Reshape((10, int(vector_size/ 10), 1))(input1)

    # First Conv2D path
    conv1 = layers.Conv2D(3, (10, 10), activation=None,dilation_rate=(1,1), padding='same', strides=(1, 1),
                          kernel_regularizer=tf.keras.regularizers.l2(0.1),kernel_initializer='he_normal')(reshaped)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = layers.Activation('leaky_relu')(conv1)
    conv1 = layers.Dropout(0.15)(conv1)
    conv1 = layers.MaxPooling2D((1, 4))(conv1)

    conv1 = layers.Conv2D(3, (10, 10), activation=None,dilation_rate=(1,1), padding='same', strides=(1, 1),
                          kernel_regularizer=tf.keras.regularizers.l2(0.1),kernel_initializer='he_normal')(conv1)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = layers.Activation('leaky_relu')(conv1)
    conv1 = layers.Dropout(0.15)(conv1)
    conv1 = layers.MaxPooling2D((1, 4))(conv1)
    conv1 = layers.Flatten()(conv1)

    #reshaped2 = layers.Reshape((1, int(fingerprint_size),1))(input2)
    # Second Conv2D path
    deep = layers.Dense(250, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(input2)
    deep = layers.BatchNormalization()(deep)
    deep = layers.Activation('leaky_relu')(deep)
    deep = layers.Dropout(0.25)(deep)

    deep = layers.Dense(90, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(deep)
    deep = layers.BatchNormalization()(deep)
    deep = layers.Activation('leaky_relu')(deep)
    deep = layers.Dropout(0.25)(deep)
    #
    # deep = layers.Dense(10, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(reshaped2)
    # deep = layers.BatchNormalization()(deep)
    # deep = layers.Activation('leaky_relu')(deep)
    # deep = layers.Dropout(0.25)(deep)
    # deep = layers.Flatten()(deep)


    # Concatenate the outputs of the parallel Conv2D paths
    concatenated = layers.Concatenate()([conv1, deep])

    # Flatten the output to feed into fully connected layers
    #flattened = layers.Flatten()(concatenated)

    # Dense (fully connected) layers for regression
    dense = layers.Dense(20, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(concatenated)
    dense = layers.BatchNormalization()(dense)
    dense = layers.Activation('leaky_relu')(dense)
    dense = layers.Dropout(0.25)(dense)

    # Output layer
    outputs = layers.Dense(output_shape * num_classes, activation='sigmoid')(dense)

    # Define the model
    model = models.Model(inputs=full_input, outputs=outputs)

    return model

#mohammednoor
def create_2D_cnn_regression_parallel_weighted(vector_size=vector_size, fingerprint_size=fingerprint_size, output_shape=1, num_classes=1):
    total_size = vector_size + fingerprint_size
    full_input = Input(shape=(total_size,), name="full_input")

    # Proper symbolic slicing using Lambda
    input1 = layers.Lambda(lambda x: x[:, :vector_size], name="grind_slice")(full_input)
    input2 = layers.Lambda(lambda x: x[:, vector_size:], name="fp_slice")(full_input)
    #inputs = full_input[:, :560]
    #input2 = full_input[:, 560:]

    # Reshape the input to a 2D image-like structure (height=1, width=600, channels=1)
    reshaped = layers.Reshape((10, int(vector_size/ 10), 1))(input1)

    # First Conv2D path
    conv1 = layers.Conv2D(3, (10, 10), activation=None,dilation_rate=(1,1), padding='same', strides=(1, 1),
                          kernel_regularizer=tf.keras.regularizers.l2(0.1),kernel_initializer='he_normal')(reshaped)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = layers.Activation('leaky_relu')(conv1)
    conv1 = layers.Dropout(0.15)(conv1)
    conv1 = layers.MaxPooling2D((1, 4))(conv1)

    conv1 = layers.Conv2D(3, (10, 10), activation=None,dilation_rate=(1,1), padding='same', strides=(1, 1),
                          kernel_regularizer=tf.keras.regularizers.l2(0.1),kernel_initializer='he_normal')(conv1)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = layers.Activation('leaky_relu')(conv1)
    conv1 = layers.Dropout(0.15)(conv1)
    conv1 = layers.MaxPooling2D((1, 4))(conv1)
    conv1 = layers.Flatten()(conv1)

    conv1 = layers.Dense(20, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(conv1)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = layers.Activation('leaky_relu')(conv1)
    conv1 = layers.Dropout(0.25)(conv1)




    # Second Conv2D path
    deep = layers.Dense(250, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(input2)
    deep = layers.BatchNormalization()(deep)
    deep = layers.Activation('leaky_relu')(deep)
    deep = layers.Dropout(0.25)(deep)

    deep = layers.Dense(90, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(deep)
    deep = layers.BatchNormalization()(deep)
    deep = layers.Activation('leaky_relu')(deep)
    deep = layers.Dropout(0.25)(deep)

    deep = layers.Dense(20, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(deep)
    deep = layers.BatchNormalization()(deep)
    deep = layers.Activation('leaky_relu')(deep)
    deep = layers.Dropout(0.25)(deep)



    #
    # deep = layers.Dense(10, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(reshaped2)
    # deep = layers.BatchNormalization()(deep)
    # deep = layers.Activation('leaky_relu')(deep)
    # deep = layers.Dropout(0.25)(deep)
    # deep = layers.Flatten()(deep)


    # Concatenate the outputs of the parallel Conv2D paths
    #concatenated = layers.Concatenate()([conv1, deep])

    # Flatten the output to feed into fully connected layers
    #flattened = layers.Flatten()(concatenated)

    # Dense (fully connected) layers for regression
    # dense = layers.Dense(20, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(concatenated)
    # dense = layers.BatchNormalization()(dense)
    # dense = layers.Activation('leaky_relu')(dense)
    # dense = layers.Dropout(0.25)(dense)
    # Merging the probability predictions using user-defined weights
       # Dense layer to output probability for each branch
    prob_branch1 = layers.Dense(1, activation='sigmoid', name="prob_branch1")(conv1)
    prob_branch2 = layers.Dense(1, activation='sigmoid', name="prob_branch2")(deep)

    # Expand the weights to match the shape of the probabilities
    branch1_weight_expanded = expand_dims(0.5, axis=0)  # Reshape to (1,)
    branch2_weight_expanded = expand_dims(0.5, axis=0)  # Reshape to (1,)
    # Apply the weights to the branch outputs
    weighted_prob_branch1 = layers.Multiply()([prob_branch1, branch1_weight_expanded])
    weighted_prob_branch2 = layers.Multiply()([prob_branch2, branch2_weight_expanded])

    # Merging the probability predictions using user-defined weights
    weighted_prob = layers.Add()([weighted_prob_branch1, weighted_prob_branch2])

   # weighted_prob = layers.Add()([layers.Multiply()([branch1_weight_expanded, 0.5]), layers.Multiply()([branch2_weight_expanded, 0.5])])

    # Output layer
    outputs = layers.Dense(output_shape * num_classes, activation='sigmoid')(weighted_prob)

    # Define the model
    model = models.Model(inputs=full_input, outputs=outputs)

    return model

#mohammednoor
def create_2D_cnn_regression_deep_branch(vector_size=vector_size, fingerprint_size=fingerprint_size, output_shape=1, num_classes=1):
    total_size = vector_size + fingerprint_size
    full_input = Input(shape=(total_size,), name="full_input")

    # Proper symbolic slicing using Lambda
    input1 = layers.Lambda(lambda x: x[:, :vector_size], name="grind_slice")(full_input)
    input2 = layers.Lambda(lambda x: x[:, vector_size:], name="fp_slice")(full_input)
    #inputs = full_input[:, :560]
    #input2 = full_input[:, 560:]

    # Reshape the input to a 2D image-like structure (height=1, width=600, channels=1)
    # reshaped = layers.Reshape((10, int(vector_size/ 10), 1))(input1)
    #
    # # First Conv2D path
    # conv1 = layers.Conv2D(3, (10, 10), activation=None,dilation_rate=(1,1), padding='same', strides=(1, 1),
    #                       kernel_regularizer=tf.keras.regularizers.l2(0.1),kernel_initializer='he_normal')(reshaped)
    # conv1 = layers.BatchNormalization()(conv1)
    # conv1 = layers.Activation('leaky_relu')(conv1)
    # conv1 = layers.Dropout(0.15)(conv1)
    # conv1 = layers.MaxPooling2D((1, 4))(conv1)
    #
    # conv1 = layers.Conv2D(3, (10, 10), activation=None,dilation_rate=(1,1), padding='same', strides=(1, 1),
    #                       kernel_regularizer=tf.keras.regularizers.l2(0.1),kernel_initializer='he_normal')(conv1)
    # conv1 = layers.BatchNormalization()(conv1)
    # conv1 = layers.Activation('leaky_relu')(conv1)
    # conv1 = layers.Dropout(0.15)(conv1)
    # conv1 = layers.MaxPooling2D((1, 4))(conv1)
    # conv1 = layers.Flatten()(conv1)

    #reshaped2 = layers.Reshape((1, int(fingerprint_size),1))(input2)
    # Second Conv2D path
    deep = layers.Dense(250, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(input2)
    deep = layers.BatchNormalization()(deep)
    deep = layers.Activation('leaky_relu')(deep)
    deep = layers.Dropout(0.25)(deep)

    deep = layers.Dense(90, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(deep)
    deep = layers.BatchNormalization()(deep)
    deep = layers.Activation('leaky_relu')(deep)
    deep = layers.Dropout(0.25)(deep)
    #
    # deep = layers.Dense(10, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(reshaped2)
    # deep = layers.BatchNormalization()(deep)
    # deep = layers.Activation('leaky_relu')(deep)
    # deep = layers.Dropout(0.25)(deep)
    # deep = layers.Flatten()(deep)


    # Concatenate the outputs of the parallel Conv2D paths
    #concatenated = layers.Concatenate()([conv1, deep])

    # Flatten the output to feed into fully connected layers
    #flattened = layers.Flatten()(concatenated)

    # Dense (fully connected) layers for regression
    dense = layers.Dense(20, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(deep)
    dense = layers.BatchNormalization()(dense)
    dense = layers.Activation('leaky_relu')(dense)
    dense = layers.Dropout(0.25)(dense)

    # Output layer
    outputs = layers.Dense(output_shape * num_classes, activation='sigmoid')(dense)

    # Define the model
    model = models.Model(inputs=full_input, outputs=outputs)

    return model

def create_2D_cnn_regression_conv_branch(vector_size=vector_size, fingerprint_size=fingerprint_size, output_shape=1, num_classes=1):
    total_size = vector_size + fingerprint_size
    full_input = Input(shape=(total_size,), name="full_input")

    # Proper symbolic slicing using Lambda
    input1 = layers.Lambda(lambda x: x[:, :vector_size], name="grind_slice")(full_input)
    input2 = layers.Lambda(lambda x: x[:, vector_size:], name="fp_slice")(full_input)
    #inputs = full_input[:, :560]
    #input2 = full_input[:, 560:]

    # Reshape the input to a 2D image-like structure (height=1, width=600, channels=1)
    reshaped = layers.Reshape((10, int(vector_size/ 10), 1))(input1)
    #
    # # First Conv2D path
    conv1 = layers.Conv2D(3, (10, 10), activation=None,dilation_rate=(1,1), padding='same', strides=(1, 1),
                          kernel_regularizer=tf.keras.regularizers.l2(0.1),kernel_initializer='he_normal')(reshaped)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = layers.Activation('leaky_relu')(conv1)
    conv1 = layers.Dropout(0.15)(conv1)
    conv1 = layers.MaxPooling2D((1, 4))(conv1)

    conv1 = layers.Conv2D(3, (10, 10), activation=None,dilation_rate=(1,1), padding='same', strides=(1, 1),
                          kernel_regularizer=tf.keras.regularizers.l2(0.1),kernel_initializer='he_normal')(conv1)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = layers.Activation('leaky_relu')(conv1)
    conv1 = layers.Dropout(0.15)(conv1)
    conv1 = layers.MaxPooling2D((1, 4))(conv1)
    conv1 = layers.Flatten()(conv1)

    #reshaped2 = layers.Reshape((1, int(fingerprint_size),1))(input2)
    # Second Conv2D path
    # deep = layers.Dense(250, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(input2)
    # deep = layers.BatchNormalization()(deep)
    # deep = layers.Activation('leaky_relu')(deep)
    # deep = layers.Dropout(0.25)(deep)
    #
    # deep = layers.Dense(90, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(deep)
    # deep = layers.BatchNormalization()(deep)
    # deep = layers.Activation('leaky_relu')(deep)
    # deep = layers.Dropout(0.25)(deep)
    #
    # deep = layers.Dense(10, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(reshaped2)
    # deep = layers.BatchNormalization()(deep)
    # deep = layers.Activation('leaky_relu')(deep)
    # deep = layers.Dropout(0.25)(deep)
    # deep = layers.Flatten()(deep)


    # Concatenate the outputs of the parallel Conv2D paths
    #concatenated = layers.Concatenate()([conv1, deep])

    # Flatten the output to feed into fully connected layers
    #flattened = layers.Flatten()(concatenated)

    # Dense (fully connected) layers for regression
    dense = layers.Dense(20, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.03))(conv1)
    dense = layers.BatchNormalization()(dense)
    dense = layers.Activation('leaky_relu')(dense)
    dense = layers.Dropout(0.25)(dense)

    # Output layer
    outputs = layers.Dense(output_shape * num_classes, activation='sigmoid')(dense)

    # Define the model
    model = models.Model(inputs=full_input, outputs=outputs)

    return model


# Create the CNN model
#model = create_cnn()
#model = create_2D_cnn_regression2()
model= create_2D_cnn_regression_deep_branch()
#model=create_2D_cnn_regression_parallel()
#model=create_2D_cnn_regression_parallel_weighted()
#model=create_2D_cnn_regression_conv_branch()
#model = create_2D_cnn()
#model=create_DNN()
# Display the model architecture
model.summary()


# Plot the model architecture and input/output shapes
plot_model(model, to_file='model_shapes.png', show_shapes=True, dpi=60)    #,rankdir="LR")

# mohammednoor 2 optimum
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2  # For optional smoothing

# Define the Y-axis labels
y_labels = ["Dry-Dry", "O-O", "N1-N1", "TIP-TIP", "Dry-O",
            "Dry-N1", "Dry-TIP", "O-N1", "O-TIP", "N1-TIP"]

def generate_saliency_map(model, input_data,filename=None):
    """
    Generate a smoothed saliency map for active predictions using the provided model and input data.

    Parameters:
    model (tf.keras.Model): Trained Keras model for predictions.
    input_data (np.array): Input feature data for which to generate the saliency map.

    Returns:
    None. Displays the smoothed saliency map plot.
    """
    # Convert NumPy array to TensorFlow tensor
    input_tensor = tf.convert_to_tensor(input_data)
    input_tensor = tf.Variable(input_tensor)  # Ensure it can be watched

    # Get model prediction and gradients
    with tf.GradientTape() as tape:
        tape.watch(input_tensor)
        prediction = model(input_tensor, training=False)  # Forward pass

    grads = tape.gradient(prediction, input_tensor)  # Compute gradients
    saliency_map = np.abs(grads.numpy()[0])  # Convert gradients to NumPy

    # Normalize for better visualization
    saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min() + 1e-8)

    # Reshape the saliency map
    saliency_map_reshaped = saliency_map.reshape(10, int(vector_size / 10))

    # **Apply Gaussian Smoothing**
    saliency_map_smoothed = cv2.GaussianBlur(saliency_map_reshaped, (3, 3), 0.1)  # Larger kernel for smoothness

    # **Apply Bilateral Filtering (Optional)**
   # saliency_map_smoothed = cv2.bilateralFilter(saliency_map_smoothed.astype(np.float32), d=9, sigmaColor=75, sigmaSpace=75)

    # Plot the saliency map with smooth interpolation
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(saliency_map_smoothed, cmap="jet", aspect="auto", interpolation='bilinear')

    # Set Y-axis labels
    ax.set_yticks(np.arange(10))  # 10 rows
    ax.set_yticklabels(y_labels)

    # Label the axes
    ax.set_xlabel("Feature Columns")
    ax.set_ylabel("Feature Groups")
    plt.title("Smoothed Saliency Map")

    # Add color bar
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)  # Shorter color bar
    cbar.set_label("Importance")
    if filename:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"Saved saliency map to {filename}")

    plt.show()
    saliency_map
    return saliency_map

def generate_saliency_map_branch(model, input_data,filename=None):
    """
    Generate a smoothed saliency map for active predictions using the provided model and input data.

    Parameters:
    model (tf.keras.Model): Trained Keras model for predictions.
    input_data (np.array): Input feature data for which to generate the saliency map.

    Returns:
    None. Displays the smoothed saliency map plot.
    """
    # Convert NumPy array to TensorFlow tensor
    input_tensor = tf.convert_to_tensor(input_data)
    input_tensor = tf.Variable(input_tensor)  # Ensure it can be watched

    # Get model prediction and gradients
    with tf.GradientTape() as tape:
        tape.watch(input_tensor)
        prediction = model(input_tensor, training=False)  # Forward pass

    grads = tape.gradient(prediction, input_tensor)  # Compute gradients
    saliency_map = np.abs(grads.numpy()[0])  # Convert gradients to NumPy

    # Normalize for better visualization
    saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min() + 1e-8)

    # Reshape the saliency map
    saliency_map_reshaped = saliency_map.reshape(10, int(vector_size / 10))

    # **Apply Gaussian Smoothing**
    saliency_map_smoothed = cv2.GaussianBlur(saliency_map_reshaped, (3, 3), 0.35)  # Larger kernel for smoothness

    # **Apply Bilateral Filtering (Optional)**
   # saliency_map_smoothed = cv2.bilateralFilter(saliency_map_smoothed.astype(np.float32), d=9, sigmaColor=75, sigmaSpace=75)

    # Plot the saliency map with smooth interpolation
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(saliency_map_smoothed, cmap="jet", aspect="auto", interpolation='bilinear')

    # Set Y-axis labels
    ax.set_yticks(np.arange(10))  # 10 rows
    ax.set_yticklabels(y_labels)

    # Label the axes
    ax.set_xlabel("Feature Columns")
    ax.set_ylabel("Feature Groups")
    plt.title("Smoothed Saliency Map")

    # Add color bar
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)  # Shorter color bar
    cbar.set_label("Importance")
    if filename:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"Saved saliency map to {filename}")

    plt.show()
    return saliency_map

def  draw_saliency_map_mean(saliency_maps):
    # Normalize for better visualization
    saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min() + 1e-8)

    # Reshape the saliency map
    saliency_map_reshaped = saliency_map.reshape(10, int(vector_size / 10))

    # **Apply Gaussian Smoothing**
    saliency_map_smoothed = cv2.GaussianBlur(saliency_map_reshaped, (3, 3), 0.35)  # Larger kernel for smoothness

    # **Apply Bilateral Filtering (Optional)**
   # saliency_map_smoothed = cv2.bilateralFilter(saliency_map_smoothed.astype(np.float32), d=9, sigmaColor=75, sigmaSpace=75)

    # Plot the saliency map with smooth interpolation
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(saliency_map_smoothed, cmap="jet", aspect="auto", interpolation='bilinear')

    # Set Y-axis labels
    ax.set_yticks(np.arange(10))  # 10 rows
    ax.set_yticklabels(y_labels)

    # Label the axes
    ax.set_xlabel("Feature Columns")
    ax.set_ylabel("Feature Groups")
    plt.title("Smoothed Saliency Map")

    # Add color bar
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)  # Shorter color bar
    cbar.set_label("Importance")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow.keras.layers as layers
from tensorflow.keras import models
#mohammednoor
# Visualize the filters of a convolutional layer with color bar
def plot_conv_weights_for_layer(idx,conv_layer, filename=None):
    filters, biases = conv_layer.get_weights()
    n_filters = filters.shape[-1]

    # Visualizing each filter
    fig, axes = plt.subplots(1, n_filters, figsize=(20, 5))
    for i in range(n_filters):
        f = filters[:, :, 0, i]
        ax = axes[i]
        im = ax.imshow(f, cmap='gray')
        ax.axis('off')

        # Add color bar
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=8)
    if filename:
            plt.savefig(f"{filename.rsplit('.', 1)[0]}_{idx}.jpeg", dpi=300, bbox_inches='tight')
            print(f"Saved conv filter map {idx} to {filename}")
    plt.show()

# Visualize filters for all convolutional layers
def plot_conv_weights_all_layers(model, filename=None):
    for idx, layer in enumerate(model.layers):
        if isinstance(layer, layers.Conv2D):
            print(f"Visualizing filters of Conv2D layer {idx}")
            plot_conv_weights_for_layer(idx,layer,filename)


# Call function to plot weights
#plot_conv_weights_all_layers(model)

# Generate random test data for activations
#test_data = np.random.rand(1, 1870)  # Single sample with the input shape
#plot_layer_activations(model, layer_idx=1, input_data=test_data)
###merge GRIND+Pubchem
#### this is used if you are using deep_branch of grind_branch models alone or in combination
#### because this make merge of two finger prints that are then splitted at first layer of model(s).
X_total = np.hstack((X, X2))
#X=X_total
#stratifiedkfold with sampling all inactives throughout iterations
# MohammedNoor2
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score, confusion_matrix, balanced_accuracy_score, matthews_corrcoef
import tensorflow as tf
from tensorflow.keras import losses
from tensorflow.keras.metrics import AUC
import csv

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# Splitting data into active and inactive classes
active_indices = np.where(y == 1)[0]
inactive_indices = np.where(y == 0)[0]

oversampling=False
if oversampling :

    # Count original active and inactive molecules
    A = len(active_indices)  # Active count
    I = len(inactive_indices)  # Inactive count

    # Calculate the required number of active samples to make them 30% of total data
    A_new = int((30 / 70) * I)  # Total required active count
    num_new_samples = max(0, A_new - A)  # Additional active molecules needed

    # Perform oversampling if needed
    if num_new_samples > 0:
        new_active_indices = resample(active_indices,
                                      replace=True,  # Sampling with replacement
                                      n_samples=num_new_samples,
                                      random_state=42)

        # Combine original active indices with new oversampled ones
        augmented_active_indices = np.hstack((active_indices, new_active_indices))
    else:
        augmented_active_indices = active_indices  # No oversampling needed

    active_indices=augmented_active_indices


# Calculate the number of active samples
n_active = len(active_indices)

# Divide inactive indices into 5 equal subsets for rotation
n_splits = 5
inactive_subsets = np.array_split(inactive_indices, n_splits)

# Prepare for metrics collection
all_metrics = {'AUC-ROC': [], 'Balanced Accuracy': [], 'MCC': [], 'TP': [],
               'TN': [], 'FP': [], 'FN': []}

import datetime
import os

stamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
stamp=str(stamp)
filename = f"file_{stamp}.txt"
#print(filename)  # Example: file_20250316123045.txt

class_labels = {
    2: 'Mu agonist',
    1: 'Kappa agonist',
    0: 'Delta agonist',
    5: 'Mu antagonist',
    4: 'Kappa antagonist',
    3: 'Delta antagonist'
    }

saliency_maps=[]
print(f"Ligands {class_labels[selected_activity]}")

folder_name = stamp
# Create the folder
os.makedirs(folder_name, exist_ok=True)


from keras.callbacks import EarlyStopping

# Define EarlyStopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',   # Monitor validation loss
    patience=20,           # Stop if val_loss doesn't improve for 20 epochs
    restore_best_weights=True  # Restore best model weights after stopping
)


with open('metrics_results.csv', mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([class_labels[selected_activity],"#" , stamp, "#", "#", "#", "#", "#", "#"])


# Add p-values storage for MWT and PSA
all_p_values_mwt = []
all_p_values_psa = []


# Number of iterations
num_iterations = 1

# Iterate over the number of iterations
for iteration in range(num_iterations):
    print(f"Iteration {iteration + 1}/{num_iterations}")

    # Rotate inactive subsets for this iteration
    current_inactive_subset = inactive_subsets[iteration % n_splits]

    # Ensure 1:2 active-to-inactive ratio
    if len(current_inactive_subset) < n_active * 2:
        inactive_to_use = resample(current_inactive_subset, n_samples=n_active * 2, random_state=iteration)
    else:
        inactive_to_use = np.random.choice(current_inactive_subset, size=n_active * 2, replace=False)

    # Combine active and inactive indices
    combined_indices = np.hstack((active_indices, inactive_to_use))
    np.random.shuffle(combined_indices)

    # Extract features and labels for this iteration
    x_balanced = X[combined_indices]
    y_balanced = y[combined_indices]
    z_balanced = z[combined_indices]
    # Stratified K-Fold setup
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=iteration)

    # To store metrics for the current iteration
    fold_metrics = {'AUC-ROC': [], 'Balanced Accuracy': [], 'MCC': [], 'TP': [],
                    'TN': [], 'FP': [], 'FN': []}


    fold = 1
    for train_index, val_index in kf.split(x_balanced, y_balanced):
        x_train, x_val = x_balanced[train_index], x_balanced[val_index]
        y_train, y_val = y_balanced[train_index], y_balanced[val_index]
        z_train, z_val = z_balanced[train_index], z_balanced[val_index]

        # Create and compile the model
        model = create_2D_cnn_regression2(input_shape=(vector_size,), output_shape=1, num_classes=1)
        #model = create_2D_cnn_regression_deep_branch(vector_size,fingerprint_size, output_shape=1, num_classes=1)
        #model = create_2D_cnn_regression_conv_branch(vector_size,fingerprint_size, output_shape=1, num_classes=1)
        #model2 = create_2D_cnn_regression_deep_branch(vector_size,fingerprint_size, output_shape=1, num_classes=1)
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss=losses.BinaryCrossentropy(), metrics=['accuracy', AUC(name='auc')])

        # Train the model
        history = model.fit(x_train, y_train, epochs=200, batch_size=5, validation_data=(x_val, y_val), verbose=1,callbacks=[early_stopping])
        #history2 = model2.fit(x_train, y_train, epochs=200, batch_size=5, validation_data=(x_val, y_val), verbose=1,callbacks=[early_stopping])
        # Predict on the validation set
        y_val_pred = model.predict(x_val).flatten()
        #y_val_pred2 = model2.predict(x_val).flatten()
        weight_conv=0.5
        weight_deep=0.5

        total = weight_conv + weight_deep
        #y_val_pred = (weight_conv * y_val_pred + weight_deep * y_val_pred2) / total
        # Calculate metrics
        auc_roc = roc_auc_score(y_val, y_val_pred)
        y_val_pred_binary = (y_val_pred > 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, y_val_pred_binary).ravel()
        balanced_acc = balanced_accuracy_score(y_val, y_val_pred_binary)
        mcc = matthews_corrcoef(y_val, y_val_pred_binary)

        failed_active_indexes=np.where((y_val==1) & (y_val == y_val_pred_binary))
        failed_inactive_indexes=np.where((y_val==0) & (y_val == y_val_pred_binary))

        failed_active_smiles=z_val[failed_active_indexes[0]]
        failed_inactive_smiles=z_val[failed_inactive_indexes[0]]



        from rdkit import Chem
        from rdkit.Chem import Draw
        print(f"Failed active smiles {failed_active_smiles}")
        # Convert SMILES to RDKit molecules
        #failed_active_molecules = [Chem.MolFromSmiles(smiles) for smiles in failed_active_smiles]
        # Display as grid (4 columns, auto-adjusted rows)
        #if len(failed_active_molecules) > 0:
            #img = Draw.MolsToGridImage(failed_active_molecules, molsPerRow=4, subImgSize=(200, 200))
            # Show the image
            #display(img)

        print(f"Failed inactive smiles {failed_inactive_smiles}")
        # Convert SMILES to RDKit molecules
        #failed_inactive_molecules = [Chem.MolFromSmiles(smiles) for smiles in failed_inactive_smiles]
        # Display as grid (4 columns, auto-adjusted rows)
        #if len(failed_inactive_molecules) > 0:
            #img2 = Draw.MolsToGridImage(failed_inactive_molecules, molsPerRow=4, subImgSize=(200, 200))
            # Show the image
            #display(img2)

        # Store metrics for the fold
        fold_metrics['AUC-ROC'].append(auc_roc)
        fold_metrics['TP'].append(tp)
        fold_metrics['TN'].append(tn)
        fold_metrics['FP'].append(fp)
        fold_metrics['FN'].append(fn)
        fold_metrics['Balanced Accuracy'].append(balanced_acc)
        fold_metrics['MCC'].append(mcc)


        # Print metrics
        print(f"Iteration {iteration + 1}, Fold {fold}, AUC-ROC: {auc_roc:.4f}, BA: {balanced_acc:.4f}, MCC: {mcc:.4f} ")
        print("Confusion Matrix:")
        print(f"             Predicted")
        print(f"             0     1")
        print(f"Actual  0    {tn:<5} {fp:<5}")
        print(f"        1    {fn:<5} {tp:<5}")

        # Write results to CSV
        with open('metrics_results.csv', mode='a', newline='') as file:
            writer = csv.writer(file)
            if iteration == 0 and fold == 1:
                writer.writerow(['Iteration', 'Fold', 'AUC-ROC', 'Balanced Accuracy', 'MCC', 'TP', 'TN', 'FP', 'FN'])
            writer.writerow([iteration + 1, fold, auc_roc, balanced_acc, mcc, tp, tn, fp, fn])

        ##only used with grind fingerprint
        filename_saliency = f"{class_labels[selected_activity]}_saliencymap_iter{iteration + 1}_fold{fold}.jpeg"
         # Save a file inside the folder
        file_path_saliency = os.path.join(folder_name, filename_saliency)
        #saliency_individual_map=generate_saliency_map(model, x_val,file_path_saliency)
        saliency_individual_map=generate_saliency_map(model,x_val,file_path_saliency)
        saliency_maps.append(saliency_individual_map)

        filename_Kernels = f"{class_labels[selected_activity]}_Kernels_iter{iteration + 1}_fold{fold}.jpeg"
        # Save a file inside the folder
        file_path_kernels = os.path.join(folder_name, filename_Kernels)
        plot_conv_weights_all_layers(model,file_path_kernels)
        #remove next two lines if want to use all 5 folds
        #if fold==2:
        #    break
        fold += 1

    # Aggregate metrics across folds
    for metric in fold_metrics:
        all_metrics[metric].extend(fold_metrics[metric])

#mad_0_1 = mean_absolute_difference(saliency_maps[0], saliency_maps[1])
#ssim_0_1 = compare_ssim(saliency_maps[0], saliency_maps[1])

# Calculate and print average metrics
mean_metrics = {metric: np.mean(values) for metric, values in all_metrics.items()}
std_metrics = {metric: np.std(values) for metric, values in all_metrics.items()}

for metric in mean_metrics:
    print(f"Average {metric}: {mean_metrics[metric]:.4f} ± {std_metrics[metric]:.4f}")

print("Training complete")


import numpy as np
import matplotlib.pyplot as plt

def plot_mean_std_saliency_maps(saliency_maps, vector_size, y_labels=None, filename=None):
    """
    Computes and plots the mean and standard deviation of saliency maps
    using the same color scale for both.

    Parameters:
    - saliency_maps (list of np.array): List of 1D saliency maps (already normalized).
    - vector_size (int): Total size of the flattened input vector.
    - y_labels (list of str): Optional Y-axis labels.
    - filename (str): Optional filename to save the plot.

    Returns:
    - None. Displays and optionally saves the saliency maps.
    """
    saliency_maps_np = np.array(saliency_maps)  # Shape: (n_maps, vector_size)
    mean_map = np.mean(saliency_maps_np, axis=0)
    std_map = np.std(saliency_maps_np, axis=0)

    # Reshape for 2D visualization
    height = 10
    width = int(vector_size / height)
    mean_map_2d = mean_map.reshape((height, width))
    std_map_2d = std_map.reshape((height, width))

    # Use default labels if not provided
    if y_labels is None:
        y_labels = ["Dry-Dry", "O-O", "N1-N1", "TIP-TIP", "Dry-O",
                    "Dry-N1", "Dry-TIP", "O-N1", "O-TIP", "N1-TIP"]

    # Compute global vmin and vmax for consistent color scaling
    combined_min = min(mean_map_2d.min(), std_map_2d.min())   # or use value like 0
    combined_max = max(mean_map_2d.max(), std_map_2d.max())    # or use value like 0.4

    # Plot both mean and std maps with the same color scale
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))

    for ax, data, title in zip(axs,
                                [mean_map_2d, std_map_2d],
                                ["Mean Saliency Map", "Standard Deviation Map"]):
        im = ax.imshow(data, cmap="jet", aspect="auto", interpolation='bilinear',
                       vmin=combined_min, vmax=combined_max)
        ax.set_yticks(np.arange(height))
        ax.set_yticklabels(y_labels)
        ax.set_xlabel("Feature Columns")
        ax.set_ylabel("Feature Groups")
        ax.set_title(title)
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Importance")

    plt.tight_layout()
    if filename:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"Saved mean ± std saliency maps to {filename}")
    plt.show()



plot_mean_std_saliency_maps(saliency_maps, 560,None,"mean_std_Delta_agonist_test")

# Assume saliency_maps is a list of arrays
saliency_maps_np = np.array(saliency_maps)  # Convert list to array
np.save("saliency_maps_Delta_agonist_test.npy", saliency_maps_np)  # Save to file
print("Saved saliency maps to saliency_maps.npy")



###in case you save and reload saliency map as numpy array
#loaded_maps = np.load("saliency_maps_Delta_agonist_test.npy")
#print("Loaded saliency maps with shape:", loaded_maps.shape)
#plot_mean_std_saliency_maps(loaded_maps, 560,None,"mean_std_Mu_antagonist2_reloaded")